# 🔬 Diagnóstico Fase 5 (v2) — a divergência é SÓ de tipo (int × float)?

A v1 provou: sondas de `random`/`math` idênticas entre CPython 3.12 e 3.13, streams MT idênticos, e o único diff no tick 1 é `remaining.<depósito>`: **int** no Python, **float** no nativo (mesmo valor). Hipótese: o hash do veredito (JSON) só diverge quando um depósito termina a corrida **sem nenhuma extração** (no Python, `remaining` nasce `int` do `size_t` e só vira `float` na primeira subtração; os testes locais comparam dicts com `==`, onde `3200000 == 3200000.0`). Esta v2 roda as 11 configurações do veredito e classifica cada diff em somente-tipo × valor, em `state_dict` e `result()`.

In [ ]:
# 1) Fonte do pacote: dataset privado do bundle (mundo-autonomo-srcN).
#    LOADER RECURSIVO (layout novo do Kaggle) — mesmo do veredito.
import glob, os, sys, zipfile

inits = sorted(glob.glob('/kaggle/input/**/src/mundo/__init__.py',
                         recursive=True))
zip_cands = sorted(glob.glob('/kaggle/input/**/mundo*.zip', recursive=True))
if inits:
    src_root = os.path.dirname(os.path.dirname(os.path.dirname(inits[-1])))
elif zip_cands:
    zipfile.ZipFile(zip_cands[-1]).extractall('/kaggle/working/pkg')
    src_root = '/kaggle/working/pkg'
else:
    raise SystemExit('Fonte do pacote nao encontrada (loader recursivo).')
print('pacote em', src_root)
sys.path.insert(0, os.path.join(src_root, 'src'))
import mundo
print('mundo', mundo.__version__, 'importado de', src_root)


In [ ]:
# 2) Compila o nucleo nativo a partir das fontes do bundle.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nanobind'],
               check=True)
subprocess.run(['g++', '--version'], check=False)
from mundo import native
from mundo.native import build
path = build.build(force=True)
print('nucleo compilado:', path)
print('native.available():', native.available())
assert native.available(), 'nucleo nativo nao carregou no kernel'


In [ ]:
# 3) COMPARACAO TIPADA nos estados FINAIS das 11 configuracoes do
#    veredito: H (regra do veredito), H_norm (floats integrais viram int
#    dos DOIS lados — igualdade aqui prova ZERO divergencia de VALOR) e
#    o inventario completo dos diffs somente-de-tipo (int x float com o
#    mesmo valor) em state_dict e em result().
import hashlib, json, sys
from collections import Counter

from mundo.engine import Engine
from mundo.native_engine import NativeEngine

print('python:', sys.version)

def H(x):
    return hashlib.sha256(json.dumps(
        x, sort_keys=True, ensure_ascii=False,
        default=str).encode()).hexdigest()

def norm(x):
    if isinstance(x, float):
        return int(x) if x.is_integer() else x
    if isinstance(x, dict):
        return {k: norm(v) for k, v in x.items()}
    if isinstance(x, list):
        return [norm(v) for v in x]
    return x

def typed_diff(a, b, path, type_only, values, limit=400):
    if len(values) >= limit:
        return
    if type(a) is not type(b):
        if (isinstance(a, (int, float)) and isinstance(b, (int, float))
                and not isinstance(a, bool) and not isinstance(b, bool)):
            if float(a) == float(b):
                type_only.append(path)
            else:
                values.append((path, repr(a), repr(b)))
            return
        values.append((path, f'{type(a).__name__}:{repr(a)[:60]}',
                       f'{type(b).__name__}:{repr(b)[:60]}'))
        return
    if isinstance(a, dict):
        for k in sorted(set(a) | set(b), key=str):
            if k not in a or k not in b:
                values.append((f'{path}.{k}', 'presente' if k in a else 'ausente',
                               'presente' if k in b else 'ausente'))
            else:
                typed_diff(a[k], b[k], f'{path}.{k}', type_only, values, limit)
    elif isinstance(a, list):
        if len(a) != len(b):
            values.append((path, f'len={len(a)}', f'len={len(b)}'))
            return
        for i, (x, y) in enumerate(zip(a, b)):
            typed_diff(x, y, f'{path}[{i}]', type_only, values, limit)
    elif a != b:
        values.append((path, repr(a), repr(b)))

def collapse(paths):
    out = Counter()
    for p in paths:
        parts = []
        for seg in p.split('.'):
            if seg.endswith(']') and '[' in seg:
                seg = seg[:seg.index('[') + 1] + '*]'
            parts.append(seg)
        out['.'.join(parts)] += 1
    return dict(out)

LIVING = dict(market_v2=True, country_v2=True, equip_v2=True)
GOD = [
    {"tick": 5, "type": "price_shock", "commodity": "copper", "factor": 1.8},
    {"tick": 10, "type": "discovery", "name": "Serra Nova",
     "resource": "lithium_ore", "lat": -20.0, "lon": -45.0,
     "size_t": 9000, "grade": 0.8},
    {"tick": 15, "type": "sanction", "resource": "copper_ore", "duration": 8},
]
FORCE = {"force_events": [
    {"tick": 3, "country": "Chile", "event": "election"},
    {"tick": 6, "country": "Brasil", "event": "export_restriction",
     "resource": "iron_ore"},
    {"tick": 9, "country": "Argentina", "event": "disaster"},
    {"tick": 12, "country": "Peru", "event": "royalty_reform"},
    {"tick": 15, "country": "Chile", "event": "state_jv"},
    {"tick": 18, "country": "Brasil", "event": "subsidy"},
    {"tick": 21, "country": "Peru", "event": "judicial_halt"},
]}
CASES = [
    ("classico n=8 t=120", 120, dict(seed=42, n_companies=8)),
    ("classico n=100 t=48", 48, dict(seed=42, n_companies=100)),
    ("vivo n=8 t=120", 120, dict(seed=42, n_companies=8, **LIVING)),
    ("vivo n=100 t=48", 48, dict(seed=42, n_companies=100, **LIVING)),
    ("vivo n=1000 t=24", 24, dict(seed=42, n_companies=1000, **LIVING)),
    ("vivo seed 7 t=60", 60, dict(seed=7, n_companies=8, **LIVING)),
    ("vivo seed 123 t=60", 60, dict(seed=123, n_companies=8, **LIVING)),
    ("vivo+deus t=60", 60, dict(seed=7, n_companies=8, god_events=GOD, **LIVING)),
    ("vivo+force t=48", 48, dict(seed=42, n_companies=8,
                                 country_params=FORCE, **LIVING)),
]

report = []
for label, ticks, kw in CASES:
    py = Engine(**kw); py.run(ticks)
    nat = NativeEngine(**kw); nat.run(ticks)
    sp, sn = py.state_dict(), nat.state_dict()
    rp, rn = py.result(), nat.result()
    to_s, val_s, to_r, val_r = [], [], [], []
    typed_diff(sp, sn, 'state', to_s, val_s)
    typed_diff(rp, rn, 'result', to_r, val_r)
    row = {
        'label': label,
        'H_state': H(sp) == H(sn), 'H_result': H(rp) == H(rn),
        'Hnorm_state': H(norm(sp)) == H(norm(sn)),
        'Hnorm_result': H(norm(rp)) == H(norm(rn)),
        'type_only_state': collapse(to_s), 'type_only_result': collapse(to_r),
        'values_state': val_s[:20], 'values_result': val_r[:20],
        'remaining_int_py': sorted(
            n for n, v in sp['remaining'].items() if isinstance(v, int)),
    }
    report.append(row)
    print(f"[{label}] H_state={row['H_state']} H_result={row['H_result']} "
          f"Hnorm_state={row['Hnorm_state']} Hnorm_result={row['Hnorm_result']}")
    print(f"    type-only state: {row['type_only_state']}")
    print(f"    type-only result: {row['type_only_result']}")
    print(f"    values state: {val_s[:5]}")
    print(f"    values result: {val_r[:5]}")
    print(f"    remaining int no Python: {row['remaining_int_py']}")

def steps(e, n):
    for _ in range(n):
        e.step()

for flags, label in [(dict(), 'classico'), (LIVING, 'vivo')]:
    ref = Engine(seed=42, n_companies=8, **flags); steps(ref, 60)
    a = Engine(seed=42, n_companies=8, **flags); steps(a, 30)
    b = NativeEngine.from_state(a.state_dict()); steps(b, 30)
    c = NativeEngine(seed=42, n_companies=8, **flags); steps(c, 30)
    d = Engine.from_state(c.state_dict()); steps(d, 30)
    sref = ref.state_dict()
    for tag, eng in [('py->nat', b), ('nat->py', d)]:
        se = eng.state_dict()
        to_s, val_s = [], []
        typed_diff(sref, se, 'state', to_s, val_s)
        row = {'label': f'restore {label} {tag}',
               'H_state': H(sref) == H(se),
               'Hnorm_state': H(norm(sref)) == H(norm(se)),
               'type_only_state': collapse(to_s),
               'values_state': val_s[:20],
               'remaining_int_py': sorted(
                   n for n, v in sref['remaining'].items()
                   if isinstance(v, int))}
        report.append(row)
        print(f"[{row['label']}] H_state={row['H_state']} "
              f"Hnorm_state={row['Hnorm_state']}")
        print(f"    type-only state: {row['type_only_state']}")
        print(f"    values state: {val_s[:5]}")


In [ ]:
# 4) Resumo do diagnostico v2.
import json

print('=' * 64)
print('DIAG2-RESUMO FASE 5:')
all_norm = True
for r in report:
    ok = r.get('Hnorm_state', True) and r.get('Hnorm_result', True)
    all_norm &= ok
    print(f"  {r['label']}: H={r['H_state']} Hnorm_ok={ok} "
          f"type_only={ {**r.get('type_only_state', {}), **r.get('type_only_result', {})} } "
          f"valores={len(r.get('values_state', [])) + len(r.get('values_result', []))}")
print('  TODAS as divergencias sao somente-de-tipo:', all_norm)
print('=' * 64)
with open('/kaggle/working/diag2-fase5.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=1, default=str)
